# Práctico 2 - Aplicación de LLMs

### Angaut, Gonzalo

Para este práctico, se pide:

1. Cargar un SLM (Small Language Model) y hacerlo funcionar a modo de pregunta respuesta como lo haria ChatGPT -> Una opcion puede ser [Qwen de Alibaba](https://huggingface.co/Qwen/Qwen3-0.6B) u otro modelo similar con < 1 Bi de parámetros.

1. Cargar el dataset elegido en el Práctico 1 e iterar cada una de las filas para generar una predicción.

1. Cargar el modelo entrenado en el Práctico 1 y hacer una comparación ¿Es mejor el LLM para clasificar? ¿Por qué?

1. Iterar parametros y prompt para ver como mejora.

1. Finetunear el SLM (Opcional).

1. Incorporar un modelo de huggingface (ej. BERT) a la comparación (Opcional).

La notebook a presentar debe ser legible incluyendo.
1. Introducción

1. Acompañar con comentarios que aporten a la interpretación de los resultados.

1. Una conclusión (breve pero no tan breve) con un resumen de lo trabajado y los resultados más representativos de acuerdo a su interpretación


In [ ]:
import json
import requests

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import re

from sklearn.model_selection import train_test_split

from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Preparamos el terreno

In [ ]:
# !pip install transformers

Importamos e instanciamos

## Introducción

En el presente trabajo práctico, exploraremos las capacidades de los Modelos de Lenguaje Pequeños (SLMs), una variante de menor escala de los Modelos de Lenguaje Grandes (LLMs), para la tarea de clasificación de sentimiento.

Al comienzo, cargaremos un SLM pre-entrenado, el Qwen3-0.6B, y lo configuraremos para operar como un chatbot interactivo en modo de pregunta y respuesta.

Posteriormente, utilizaremos el dataset de reseñas de películas IMDB (empleado en el Práctico 1) para realizar una clasificación de sentimiento en modo zero-shot y few-shot, utilizando técnicas de Prompt Engineering y Generación Determinística.

El objetivo central de esta práctica es comparar y analizar el rendimiento del SLM frente al modelo tradicional de TF-IDF entrenado en el Práctico 1. Además, se iterarán los parámetros de inferencia (num_beams, max_new_tokens) y el diseño del prompt para evaluar su impacto en la precisión, robustez y velocidad de la clasificación.

## Desarrollo del trabajo

Para este trabajo, vamos a utilizar el [Qwen de Alibaba](https://huggingface.co/Qwen/Qwen3-0.6B), una familia de Modelos de Lenguaje Grandes (LLMs) y Modelos de Lenguaje Pequeños (SLMs) desarrollados por Alibaba Cloud.

Cargamos este SLM:

In [ ]:
model_name = "Qwen/Qwen3-0.6B"

# Cargamos el tokenizer y el modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

# Vemos el dispositivo que estamos usando
device = model.device
print(f"Modelo cargado en el dispositivo: {device}")

A este modelo lo vamos a hacer funcionar a modo de pregunta respuesta como lo haria ChatGPT.

In [ ]:
def responder_mensajes(messages, model, tokenizer):
  # Aplicar plantilla de chat
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  # Tokenizar entrada
  inputs = tokenizer([text], return_tensors="pt").to(model.device)

  # Generar respuesta
  outputs = model.generate(
      **inputs,
      max_new_tokens=256,
      do_sample=True,# En false es deterministico
      top_p=0.9,
      temperature=0.7,
      pad_token_id=tokenizer.eos_token_id # Para evitar warnings
  )

  # Decodificar solo la parte nueva generada
  response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

  # Vamos a usar expresiones para eliminar el bloque <think>
  cleaned_response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

  # Devolvemos el texto limpio para que pueda agregarse al historial si es un chat interactivo
  return cleaned_response

Entonces, podemos hablar con el chat definiendo un mensaje:

In [ ]:
# Prompt estilo chat
messages = [
    {"role": "user", "content": "Dame una breve introducción a los modelos de lenguaje grandes."}
]

respuesta = responder_mensajes(messages, model, tokenizer)
print("Assistant:", respuesta)

Podemos hacer para que sea un chat interactivo:

In [ ]:
def chat_interactivo(model, tokenizer):
    print("CHAT")
    print("Escribe 'salir' para terminar la conversación.")

    # Inicializamos una lista de mensajes para mantener el historial
    # Esto permite que el modelo "recuerde" conversaciones anteriores.
    # Además, le vamos a decir que conteste en español
    conversation_history = [
        {"role": "system", "content": "Eres un asistente de inteligencia artificial que siempre debe responder en ESPAÑOL."}
    ]

    while True:
        user_input = input("You: ")

        if user_input.lower() == 'salir':
            print("Chat terminado.")
            break

        # Añadimos el nuevo mensaje del usuario al historial
        conversation_history.append({"role": "user", "content": user_input})

        # Llamamos a la función, pasándole el historial completo
        assistant_response = responder_mensajes(conversation_history, model, tokenizer)

        # Añadimos la respuesta limpia del asistente al historial para mantener el contexto
        conversation_history.append({"role": "assistant", "content": assistant_response})

        print("Assistant:", assistant_response)

Y entonces si llamamos a la función, podemos tener una conversación:

In [ ]:
chat_interactivo(model, tokenizer)

Ahora vamos a cargar el dataset elegido en el práctico 1, que fue el de IMDB:

In [ ]:
# Remueve archivo anterior (si existe)
!rm -f imdb_dataset.csv

# Usa wget para descarga
file_id = "1hK3hjvuoVUUV_kW8FyVE33iGig9wN0K3"
!wget --no-check-certificate "https://docs.google.com/uc?export=download&id={file_id}" -O imdb_dataset.csv

# Carga el dataset
with open("imdb_dataset.csv", 'r') as file:
    df_imdb = pd.read_csv(file)
    print(f"Success! Dataset loaded with {len(df_imdb)} records.")

In [ ]:
df = df_imdb
df.head()

Veamos, por ejemplo, una crítica:

In [ ]:
df['review'][0]

Ahora debemos ver cómo generar una predicción. Para eso, debemos armar un prompt.

Vamos a hacer una función que clasifique texto usando el SLM.

In [ ]:
def classify_slm(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS (FEW-SHOT) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Veamos por ejemplo como clasifica algun comentario:

In [ ]:
i=10
text_to_classify = df['review'][i]
prediction = classify_slm(text_to_classify, model, tokenizer, device)

print(f"Comentario (Índice {i}): {text_to_classify[:100]}...")
print(f"Predicción estable del SLM: {prediction}")
print(f"Etiqueta real: {df['sentiment'][i]}")

Podemos notar como este comentario en particular lo clasifica correctamente!

Ahora iteremos y hagamos sobre muchas filas, en particular elegiremos 1000 filas al azar. Para esto, creemos una función:

In [ ]:
def iterate_classification(df, classify_fn, model, tokenizer, device, n_samples=1000):

  # Creamos un dataset con n_samples muestras
  df_sample = df.sample(n=n_samples, random_state=42).copy()
  predictions = []

  print(f"Iniciando la predicción del SLM para {n_samples} muestras...")

  # Iteramos sobre el DataFrame de muestra con TQDM para seguimiento
  for index, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
      review_text = row['review']

      # Llamamos a la función de clasificación
      prediction = classify_fn(review_text, model, tokenizer, device)
      predictions.append(prediction)

  # Agregamos las predicciones al DataFrame de muestra
  df_sample['prediction'] = predictions

  return df_sample

Y ahora llamamos a la función:

In [ ]:
df_sample = iterate_classification(df, classify_slm, model, tokenizer, device, n_samples=1000)

Veamos algunas críticas junto con el sentimiento y las predicciones generadas:

In [ ]:
print("\n--- Predicciones del SLM Generadas ---")
print(df_sample[['review', 'sentiment', 'prediction']].sample(10))

Por último, calculemos las métricas. Para eso armamos la función:

In [ ]:
def evaluate_slm_predictions(df, true_col='sentiment', pred_col='prediction', undefined_label='INDEFINIDO', verbose=True):
    # Filtramos los casos válidos
    df_cleaned = df[df[pred_col] != undefined_label].copy()

    # Convertimos etiquetas reales a minúsculas
    y_true = df_cleaned[true_col].apply(lambda x: x.lower())
    y_pred = df_cleaned[pred_col]

    # Métricas de conteo
    total_samples = len(df)
    valid_samples = len(df_cleaned)
    failed_samples = total_samples - valid_samples

    # Calculamos métricas
    acc = accuracy_score(y_true, y_pred)
    report_dict = classification_report(y_true, y_pred, output_dict=True)
    f1_score_slm = report_dict['weighted avg']['f1-score']

    # Mostramos resultados
    if verbose:
        print(f"Total de muestras evaluadas: {total_samples}")
        print(f"Muestras con FALLO de instrucción ('{undefined_label}'): {failed_samples} ({failed_samples / total_samples * 100:.2f}%)")
        print("-" * 35)
        print(f"Precisión (Accuracy): {acc:.4f}")
        print(f"F1-score ponderado: {f1_score_slm:.4f}")
        print("\nReporte completo:")
        print(classification_report(y_true, y_pred, digits=4))

    # Retornamos resultados como diccionario
    return {
        "accuracy": acc,
        "f1_weighted": f1_score_slm,
        "total_samples": total_samples,
        "valid_samples": valid_samples,
        "failed_samples": failed_samples,
        "report": report_dict
    }

Y ahora llamamos a la función:

In [ ]:
results = evaluate_slm_predictions(df_sample)

## Comparación con resultados previos

Si recordamos del trabajo anterior, el enfoque de representación que mejor andaba en la clasificación posterior era TF-IDF con una precisión en el conjunto de testeo de $\approx 0.90$ y un F1 de $\approx 0.89$. Estos resultados son mejores que los obtenidos por el modelo base con SLM, aunque este igualmente clasifica con unas métricas bastante altas. Por lo tanto, SLM, sin entrenamiento, es ligeramente inferior a las técnicas tradicionales, pero al iterar prompts y parámetros y hacer fine-tuning, esto debería mejorar.

## Iteración de parámetros y prompts

Ahora iteremos los parámetros y prompts a ver si conseguimos mejoras.

Por ejemplo, agreguemos más ejemplos.

In [ ]:
def classify_slm_2(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS EXTENDIDOS (4 Ejemplos) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"

            "Comentario: 'Una pérdida de tiempo, el final no tiene sentido.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Recomiendo verla a todo el mundo, me hizo sentir feliz.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [ ]:
df_sample_2 = iterate_classification(df, classify_slm_2, model, tokenizer, device, n_samples=1000)

Y a la función para evaluar los resultados:

In [ ]:
results_2 = evaluate_slm_predictions(df_sample_2)

Podemos ver que al agregar más ejemplos, el modelo clasifica mejor. Podemos ver también lo que sucede si no agregamos ejemplos:

In [ ]:
def classify_slm_3(text_input, model, tokenizer, device):
    # Diseñamos el prompt, sin ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [ ]:
df_sample_3 = iterate_classification(df, classify_slm_3, model, tokenizer, device, n_samples=1000)

Y a la función para evaluar los resultados:

In [ ]:
results_3 = evaluate_slm_predictions(df_sample_3)

Podemos ver que al no agregar los ejemplos, las métricas decrecen muchísimo, lo que nos muestra la importancia de los ejemplos en el prompt.

Ahora cambiemos algunos parámetros. Por ejemplo, cambiamos `num_beams` de 5 a 10. Esto es, ahora se consideran las 10 secuencias más probables en cada paso, teniendo mayor robustez en la clasificación pero haciendo más lento el entrenamiento.

In [ ]:
def classify_slm_4(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS EXTENDIDOS (4 Ejemplos) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"

            "Comentario: 'Una pérdida de tiempo, el final no tiene sentido.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Recomiendo verla a todo el mundo, me hizo sentir feliz.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        num_beams=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [ ]:
df_sample_4 = iterate_classification(df, classify_slm_4, model, tokenizer, device, n_samples=1000)

Y a la función para evaluar los resultados:

In [ ]:
results_4 = evaluate_slm_predictions(df_sample_4)

Podemos ver que los resultados son los mismos que para `num_beams=5`, por lo que nos quedamos con ese valor.

Por último, cambiemos el `max_new_tokens` de 2 a 5, buscando mejorar la robustez de la clasificación al darle al modelo más margen para generar la etiqueta completa, lo que es crucial para reducir los fallos 'INDEFINIDO'.

In [ ]:
def classify_slm_5(text_input, model, tokenizer, device):
    # Diseñamos el prompt, con ejemplos
    prompt = (
            "Eres un clasificador de sentimiento BINARIO. La respuesta debe ser SÓLO una de dos etiquetas: 'POSITIVE' o 'NEGATIVE'. "
            "NO uses otras palabras.\n\n"

            # --- EJEMPLOS EXTENDIDOS (4 Ejemplos) ---
            "Comentario: 'La película es aburrida, predecible y lenta.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Las actuaciones fueron increíbles y el guion magistral.'\n"
            "Sentimiento: POSITIVE\n\n"

            "Comentario: 'Una pérdida de tiempo, el final no tiene sentido.'\n"
            "Sentimiento: NEGATIVE\n\n"

            "Comentario: 'Recomiendo verla a todo el mundo, me hizo sentir feliz.'\n"
            "Sentimiento: POSITIVE\n\n"
            # ----------------------------

            # El texto que queremos clasificar
            f"Comentario: {text_input}\n"
            "Sentimiento: "
        )

    # Tokenizamos
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    # Generamos
    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        num_beams=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decodificamos
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().upper()

    # Validamos
    if 'POSITIVE' in response:
        return 'positive'
    elif 'NEGATIVE' in response:
        return 'negative'
    else:
        # Devuelve un fallo para poder filtrar los errores de clasificación
        return 'INDEFINIDO'

Llamamos a la función para iterar y clasificar:

In [ ]:
df_sample_5 = iterate_classification(df, classify_slm_5, model, tokenizer, device, n_samples=1000)

Y a la función para evaluar los resultados:

In [ ]:
results_5 = evaluate_slm_predictions(df_sample_5)

## Comparación de todos los modelos

In [ ]:
# Creamos un diccionario
precision_comparison = {
    'Method': ['Initial Prompt (2 Examples)', 'Extended Prompt (4 Examples)', 'No Examples', 'Extended Prompt (num_beams=10)', 'Extended Prompt (max_new_tokens=5)'],
    'Accuracy': [results['accuracy'], results_2['accuracy'], results_3['accuracy'], results_4['accuracy'], results_5['accuracy']],
    'F1-score (weighted)': [results['f1_weighted'], results_2['f1_weighted'], results_3['f1_weighted'], results_4['f1_weighted'], results_5['f1_weighted']],
    'Failed Samples (%)': [results['failed_samples']/results['total_samples']*100,
                           results_2['failed_samples']/results_2['total_samples']*100,
                           results_3['failed_samples']/results_3['total_samples']*100,
                           results_4['failed_samples']/results_4['total_samples']*100,
                           results_5['failed_samples']/results_5['total_samples']*100]
}

# Lo pasamos a un df
df_comparison = pd.DataFrame(precision_comparison)

# Mostramos la tabla de comparación
print("--- Comparación de Resultados de Clasificación del SLM ---")
display(df_comparison)

## Conclusión

Podemos comparar lo realizado en el trabajo anterior con el modelo clásico basado en TF-IDF y el SLM Qwen-0.6B (mediante Prompt Engineering).

El modelo de TF-IDF con un clasificador tradicional se confirma como el claro ganador en rendimiento puro, logrando una Precision de $\approx0.90$ y un F1-Score de $\approx0.89$.

El SLM Qwen-0.6B demostro un rendimiento muy bueno, alcanzando un Accuracy maximo de $0.8029$ (con el Extended Prompt), a pesar de haber recibido cero entrenamiento especifico (fine-tuning) en el dataset. Observamos lo importante que es pasar de un Zero-Shot a un Few-Shot Prompting, agregando ejemplos. Esto nos dice que si seguimos mejorando el prompt, capaz podemos llegar a mejores resultados.

Los ajustes como el aumento del numero de ejemplos (Extended Prompt) y el margen de error (max_new_tokens=5) mejoraron la robustez, mientras que la iteracion de num_beams confirmo que 5 es un valor eficiente para este SLM.

Por lo tanto, en este caso, el SLM base no es mejor que el modelo tradicional de TF-IDF.

Sin embargo, el 80% de Accuracy obtenido mediante solo Prompt Engineering es una prueba del poder de su comprension contextual general.

Para que el SLM supere al modelo de TF-IDF, el siguiente paso debe ser la especializacion a traves del fine-tuning. Al entrenar el Qwen-0.6B con el dataset de IMDB, se eliminarán los fallos de instruccion y se adapptaría el modelo a la base de datos particular, seguramente maximizando el rendimiento del modelo.